## This is some examples to use this package


In [1]:
import Nbody
from Nbody import Integrator_no_grad, Integrator, State, get_default_ICs
import jax
print(jax.__version__, jax.__file__)
print(jax.devices())

0.6.2 /home/danqiu/anaconda3/envs/myenv_conda/lib/python3.13/site-packages/jax/__init__.py
[CpuDevice(id=0)]


In [2]:
import logging
import time

jax.config.update("jax_log_compiles", True)

logger = logging.getLogger("jax")
logger.setLevel(logging.WARNING)

for handler in logger.handlers[:]:
    logger.removeHandler(handler)

logger.propagate = False

handler = logging.FileHandler("jax_compile.log", mode="w")
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))

logger.addHandler(handler)

In [3]:
ic = get_default_ICs("trappist-1")
print('-------- Initial conditions, test -------')
print(ic.elements)
s = State.from_ic(ic)
print('-------- State, test -------')
print(s.v)

-------- Initial conditions, test -------
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 2.59011360e-05  1.51088006e+00  7.25754749e+03  2.43665177e-02
   1.81698840e-02  1.57079633e+00  0.00000000e+00]
 [ 5.78712551e-05  2.42180136e+00  7.25859216e+03  2.00608107e-02
   1.11897051e-02  1.57079633e+00  0.00000000e+00]
 [ 1.46027728e-06  4.05035424e+00  7.25702386e+03  7.41149016e-03
  -2.01642487e-02  1.57079633e+00  0.00000000e+00]
 [ 1.92353282e-05  6.09928159e+00  7.25781677e+03  1.18019388e-03
   7.31913418e-04  1.57079633e+00  0.00000000e+00]
 [ 2.73026874e-05  9.20618481e+00  7.25712289e+03 -6.99952921e-05
   2.25251937e-04  1.57079633e+00  0.00000000e+00]
 [ 3.53310170e-05  1.23539887e+01  7.25766733e+03 -9.72202658e-04
   1.27600040e-03  1.57079633e+00  0.00000000e+00]
 [ 1.64106270e-06  1.87335351e+01  7.25052423e+03 -1.04023031e-02
  -1.42898702e-02  1.57079633e+00  0.00000000e+00]]
-------- State, test 

In [4]:
ic = get_default_ICs("trappist-1")

s = State.from_ic(ic)
tt = Nbody.create_transit_timing(
    tmax=1000.0,
    ic=ic
)

Intr = Integrator_no_grad.default(
    h=0.1,
    t0=0.0,
    tmax=1000.0
)

start = time.perf_counter()

s, tt = Intr(s, tt)

jax.block_until_ready((s, tt))

print("Total time for Integrator without grad:", time.perf_counter() - start)

Total time for Integrator without grad: 4.568486717995256


In [5]:
with open("jax_compile.log") as f:
    for line in f:
        if (
            "Finished XLA compilation" in line
            or "Finished jaxpr to MLIR" in line
        ):
            print(line.strip())

2026-10-10 14:46:07,821 Finished jaxpr to MLIR module conversion jit(convert_element_type) in 0.005372524 sec
2026-10-10 14:46:07,826 Finished XLA compilation of jit(convert_element_type) in 0.004912615 sec
2026-10-10 14:46:07,829 Finished jaxpr to MLIR module conversion jit(dynamic_slice) in 0.001839876 sec
2026-10-10 14:46:07,838 Finished XLA compilation of jit(dynamic_slice) in 0.008423567 sec
2026-10-10 14:46:07,840 Finished jaxpr to MLIR module conversion jit(squeeze) in 0.001371861 sec
2026-10-10 14:46:07,843 Finished XLA compilation of jit(squeeze) in 0.002540112 sec
2026-10-10 14:46:07,845 Finished jaxpr to MLIR module conversion jit(convert_element_type) in 0.001164913 sec
2026-10-10 14:46:07,847 Finished XLA compilation of jit(convert_element_type) in 0.002203703 sec
2026-10-10 14:46:07,850 Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.001729250 sec
2026-10-10 14:46:07,857 Finished XLA compilation of jit(broadcast_in_dim) in 0.006561041 sec
2026-10-10 14

In [6]:
ic = get_default_ICs("trappist-1")
tmax = 1000.0
Intr = Integrator_no_grad.default(h=0.1, t0=0.0, tmax=tmax)

for i in range(3):
    s = State.from_ic(ic)
    tt = Nbody.create_transit_timing( tmax=tmax, ic=ic)
    start = time.perf_counter()
    s, tt = Intr(s, tt)
    jax.block_until_ready(s)
    jax.block_until_ready(tt)
    end = time.perf_counter()
    print(f"ReRun {i+1}: {end-start:.4f} s")

ReRun 1: 1.5022 s
ReRun 2: 1.4665 s
ReRun 3: 1.4940 s


In [7]:
tt.tt

Array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.79053962,  2.30223396,  3.81326796, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.45349103,  2.87488048,  5.29674126, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 2.64920503, 11.85743305, 21.06523851, ...,  0.        ,
         0.        ,  0.        ],
       [ 5.87594408, 18.2273673 , 30.57983533, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.64619779, 19.37745733, 38.11249612, ...,  0.        ,
         0.        ,  0.        ]], dtype=float64)

### With grad

In [8]:
ic = get_default_ICs("trappist-1")
print('-------- Initial conditions, test -------')
print(ic.elements)
s = State.from_ic(ic)
print('-------- State, test -------')
print(s.v)

-------- Initial conditions, test -------
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 2.59011360e-05  1.51088006e+00  7.25754749e+03  2.43665177e-02
   1.81698840e-02  1.57079633e+00  0.00000000e+00]
 [ 5.78712551e-05  2.42180136e+00  7.25859216e+03  2.00608107e-02
   1.11897051e-02  1.57079633e+00  0.00000000e+00]
 [ 1.46027728e-06  4.05035424e+00  7.25702386e+03  7.41149016e-03
  -2.01642487e-02  1.57079633e+00  0.00000000e+00]
 [ 1.92353282e-05  6.09928159e+00  7.25781677e+03  1.18019388e-03
   7.31913418e-04  1.57079633e+00  0.00000000e+00]
 [ 2.73026874e-05  9.20618481e+00  7.25712289e+03 -6.99952921e-05
   2.25251937e-04  1.57079633e+00  0.00000000e+00]
 [ 3.53310170e-05  1.23539887e+01  7.25766733e+03 -9.72202658e-04
   1.27600040e-03  1.57079633e+00  0.00000000e+00]
 [ 1.64106270e-06  1.87335351e+01  7.25052423e+03 -1.04023031e-02
  -1.42898702e-02  1.57079633e+00  0.00000000e+00]]
-------- State, test 

In [9]:
t0 = 0.0        # Initial time
h = 0.1         # Time step
tmax = 1000.0 # Time to integrate to.
tt = Nbody.create_transit_timing(tmax = tmax, ic = ic,)

#with grad
Intr = Integrator.default(h = h, t0 = t0, tmax = tmax)
start = time.perf_counter()
s,tt = Intr(s, tt)
end = time.perf_counter()
print(f'Running time: {end - start:.4f} seconds!')

Running time: 11.3803 seconds!


In [10]:
with open("jax_compile.log") as f:
    for line in f:
        if ("Finished XLA compilation" in line or "Finished jaxpr to MLIR" in line):
            print(line.strip())

2026-10-10 14:46:07,821 Finished jaxpr to MLIR module conversion jit(convert_element_type) in 0.005372524 sec
2026-10-10 14:46:07,826 Finished XLA compilation of jit(convert_element_type) in 0.004912615 sec
2026-10-10 14:46:07,829 Finished jaxpr to MLIR module conversion jit(dynamic_slice) in 0.001839876 sec
2026-10-10 14:46:07,838 Finished XLA compilation of jit(dynamic_slice) in 0.008423567 sec
2026-10-10 14:46:07,840 Finished jaxpr to MLIR module conversion jit(squeeze) in 0.001371861 sec
2026-10-10 14:46:07,843 Finished XLA compilation of jit(squeeze) in 0.002540112 sec
2026-10-10 14:46:07,845 Finished jaxpr to MLIR module conversion jit(convert_element_type) in 0.001164913 sec
2026-10-10 14:46:07,847 Finished XLA compilation of jit(convert_element_type) in 0.002203703 sec
2026-10-10 14:46:07,850 Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.001729250 sec
2026-10-10 14:46:07,857 Finished XLA compilation of jit(broadcast_in_dim) in 0.006561041 sec
2026-10-10 14

In [11]:
ic = get_default_ICs("trappist-1")

Intr = Integrator.default(h=0.1, t0=0.0, tmax=tmax)

for i in range(3):
    s = State.from_ic(ic)
    tt = Nbody.create_transit_timing( tmax=tmax, ic=ic)
    start = time.perf_counter()
    s, tt = Intr(s, tt)
    jax.block_until_ready(s)
    jax.block_until_ready(tt)
    end = time.perf_counter()
    print(f"ReRun {i+1}: {end-start:.4f} s")                     

ReRun 1: 5.7441 s
ReRun 2: 5.7220 s
ReRun 3: 5.7339 s


In [12]:
print('count:', tt.count)
print('tt:', tt.tt)
print('ntt:', tt.ntt)
print('ti:', tt.ti)
print('occs:', tt.occs)
print('gsave: ', tt.gsave)


count: [  0 662 413 247 163 109  81  54]
tt: [[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.79053962  2.30223396  3.81326796 ...  0.          0.
   0.        ]
 [ 0.45349103  2.87488048  5.29674126 ...  0.          0.
   0.        ]
 ...
 [ 2.64920503 11.85743305 21.06523851 ...  0.          0.
   0.        ]
 [ 5.87594408 18.2273673  30.57983533 ...  0.          0.
   0.        ]
 [ 0.64619779 19.37745733 38.11249612 ...  0.          0.
   0.        ]]
ntt: 665
ti: 0
occs: (1, 2, 3, 4, 5, 6, 7)
gsave:  [ 0.00000000e+00  6.54022068e-08 -1.89322981e-04 -7.19096202e-05
 -6.22483636e-04 -2.53470160e-03 -1.38846304e-03 -2.50025855e-03]
